## 1. Windows Local Setup Cell

Run this cell before creating `SparkSession`.

This cell is important for your Windows local setup because it helps PySpark use the correct Python environment and localhost settings.


In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"


## 2. Import SparkSession

`SparkSession` is the starting point for working with PySpark.


In [2]:
from pyspark.sql import SparkSession

## 3. Create SparkSession

We will use the same Windows-safe SparkSession configuration.


In [3]:
spark = SparkSession.builder \
    .appName("pyspark") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 4. Create Sample DataFrame

We will create a small sales dataset with some duplicate values.

Duplicates are intentionally added so that we can understand `distinct()` and `dropDuplicates()` clearly.


In [4]:
data = [
    (1, "Amit", "Delhi", "Laptop", 50000),
    (2, "Priya", "Mumbai", "Mobile", 25000),
    (3, "Rahul", "Delhi", "Laptop", 50000),
    (4, "Sneha", "Pune", "Tablet", 18000),
    (5, "Neha", "Mumbai", "Mobile", 25000),
    (6, "Karan", "Delhi", "Camera", 30000),
    (7, "Amit", "Delhi", "Laptop", 50000),
    (8, "Priya", "Mumbai", "Mobile", 25000)
]

columns = ["order_id", "customer_name", "city", "product", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

+--------+-------------+------+-------+------+
|order_id|customer_name|  city|product|amount|
+--------+-------------+------+-------+------+
|       1|         Amit| Delhi| Laptop| 50000|
|       2|        Priya|Mumbai| Mobile| 25000|
|       3|        Rahul| Delhi| Laptop| 50000|
|       4|        Sneha|  Pune| Tablet| 18000|
|       5|         Neha|Mumbai| Mobile| 25000|
|       6|        Karan| Delhi| Camera| 30000|
|       7|         Amit| Delhi| Laptop| 50000|
|       8|        Priya|Mumbai| Mobile| 25000|
+--------+-------------+------+-------+------+



## 5. Check Schema

Before applying commands, let us check the structure of the DataFrame.


In [5]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: long (nullable = true)



## 6. `withColumnRenamed()` Command

`withColumnRenamed()` is used to rename a column.

Syntax:

```python
df.withColumnRenamed("old_column_name", "new_column_name")
```

Simple meaning:

```text
withColumnRenamed() = rename column
```


In [ ]:
df_renamed = df.withColumnRenamed("customer_name", "customer")

df_renamed.show()

## 7. Rename Multiple Columns

To rename multiple columns, we can chain `withColumnsRenamed()`.


In [6]:
df_renamed_multiple = df.withColumnsRenamed({
    "customer_name": "customer",
    "amount": "sales_amount"
})

df_renamed_multiple.show()

+--------+--------+------+-------+------------+
|order_id|customer|  city|product|sales_amount|
+--------+--------+------+-------+------------+
|       1|    Amit| Delhi| Laptop|       50000|
|       2|   Priya|Mumbai| Mobile|       25000|
|       3|   Rahul| Delhi| Laptop|       50000|
|       4|   Sneha|  Pune| Tablet|       18000|
|       5|    Neha|Mumbai| Mobile|       25000|
|       6|   Karan| Delhi| Camera|       30000|
|       7|    Amit| Delhi| Laptop|       50000|
|       8|   Priya|Mumbai| Mobile|       25000|
+--------+--------+------+-------+------------+



## 8. Important Point

The original DataFrame does not change.

PySpark DataFrames are immutable.

Simple meaning:

```text
PySpark creates a new DataFrame after transformation.
Original DataFrame remains the same.
```


In [ ]:
df.show()

## 9. `drop()` Command

`drop()` is used to remove columns from a DataFrame.

Syntax:

```python
df.drop("column_name")
```

Simple meaning:

```text
drop() = remove column
```


In [ ]:
df_drop_city = df.drop("city")

df_drop_city.show()

## 10. Drop Multiple Columns

We can pass multiple column names inside `drop()`.


In [ ]:
df_drop_multiple = df.drop("city", "product")

df_drop_multiple.show()

## 11. `distinct()` Command

`distinct()` removes fully duplicate rows.

Simple meaning:

```text
distinct() = remove duplicate rows when all column values are same
```

Important:

If even one column value is different, Spark will not treat that row as duplicate.


In [ ]:
df.distinct().show()

## 12. Create DataFrame with Full Duplicate Rows

Now let us create another DataFrame where some rows are exactly the same.


In [7]:
data_duplicate = [
    (1, "Amit", "Delhi", "Laptop", 50000),
    (1, "Amit", "Delhi", "Laptop", 50000),
    (2, "Priya", "Mumbai", "Mobile", 25000),
    (2, "Priya", "Mumbai", "Mobile", 25000),
    (3, "Rahul", "Delhi", "Laptop", 50000)
]

columns_duplicate = ["order_id", "customer_name", "city", "product", "amount"]

df_duplicate = spark.createDataFrame(data_duplicate, columns_duplicate)

df_duplicate.show()

+--------+-------------+------+-------+------+
|order_id|customer_name|  city|product|amount|
+--------+-------------+------+-------+------+
|       1|         Amit| Delhi| Laptop| 50000|
|       1|         Amit| Delhi| Laptop| 50000|
|       2|        Priya|Mumbai| Mobile| 25000|
|       2|        Priya|Mumbai| Mobile| 25000|
|       3|        Rahul| Delhi| Laptop| 50000|
+--------+-------------+------+-------+------+



## 14. Apply `distinct()` on Fully Duplicate Rows

Now Spark will remove duplicate rows because all column values are exactly the same.


In [ ]:
df_duplicate.distinct().show()

## 15. `dropDuplicates()` Command

`dropDuplicates()` is also used to remove duplicate rows.

If we use it without column names, it works like `distinct()`.


In [ ]:
df_duplicate.dropDuplicates().show()

## 16. Drop Duplicates Based on Specific Columns

This is a very useful feature.

Syntax:

```python
df.dropDuplicates(["column_1", "column_2"])
```

Simple meaning:

```text
Remove duplicates by checking selected columns only.
```


In [8]:
df_duplicate.dropDuplicates(["customer_name", "product", "amount"]).show()

+--------+-------------+------+-------+------+
|order_id|customer_name|  city|product|amount|
+--------+-------------+------+-------+------+
|       1|         Amit| Delhi| Laptop| 50000|
|       2|        Priya|Mumbai| Mobile| 25000|
|       3|        Rahul| Delhi| Laptop| 50000|
+--------+-------------+------+-------+------+



## 17. Difference Between `distinct()` and `dropDuplicates()`

| Command | Meaning |
|---|---|
| `distinct()` | Removes duplicate rows by checking all columns |
| `dropDuplicates()` | Removes duplicate rows by checking all columns |
| `dropDuplicates(["col1", "col2"])` | Removes duplicates by checking selected columns only |


## 18. `orderBy()` Command

`orderBy()` is used to sort data.

Syntax:

```python
df.orderBy("column_name")
```

By default, sorting is in ascending order.


In [ ]:
df.orderBy("amount").show()

## 19. Sort in Descending Order

To sort in descending order, use `.desc()`.

For this, we use `col()`.


In [ ]:
from pyspark.sql.functions import col

In [ ]:
df.orderBy(col("amount").desc()).show()

## 20. Sort in Ascending Order

To sort in ascending order, use `.asc()`.


In [ ]:
df.orderBy(col("amount").asc()).show()

## 21. Sort by Multiple Columns

We can sort by more than one column.

Example:

```text
First sort by city in ascending order
Then sort by amount in descending order
```


In [ ]:
df.orderBy(col("city").asc(), col("amount").desc()).show()

## 22. `sort()` Command

`sort()` also sorts the DataFrame.

In most common use cases, `sort()` and `orderBy()` behave similarly.


In [ ]:
df.sort("amount").show()

##  Stop SparkSession

At the end of practice, stop SparkSession to release resources.


In [ ]:
spark.stop()